<div style="background: linear-gradient(135deg, #1F3864 0%, #2E5FA3 100%); padding: 48px 40px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; font-size: 2.4em; font-weight: 800; margin: 0 0 8px 0; letter-spacing: -0.5px;">Deep Learning for Business Analytics</h1>
  <h2 style="color: #a8c4e8; font-size: 1.3em; font-weight: 400; margin: 0 0 16px 0; font-style: italic;">From Basics to Large Language Models</h2>
  <p style="color: #c8ddf5; font-size: 0.95em; margin: 0 0 24px 0;">Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter</p>
  <div style="background: rgba(255,255,255,0.12); border-radius: 8px; padding: 16px 20px; display: inline-block;">
    <span style="color: #ddeeff; font-size: 1.05em; font-weight: 600;">&#128214; Chapter 6 &nbsp;&middot;&nbsp; Sequences, RNNs, and LSTMs</span>
  </div>
</div>
<div style="background: #f0f4fa; border-left: 5px solid #2E5FA3; padding: 14px 20px; border-radius: 0 8px 8px 0; margin-top: 4px; color: #333; font-size: 0.97em;">
  <em>How neural networks read sequences &mdash; the hidden state idea, why vanilla RNNs forget too quickly, how LSTMs fix this with gating, and a full end-to-end forecasting application on real atmospheric CO&#8322; data.</em>
</div>

## What This Chapter Covers

| Section | Topics |
|---------|--------|
| Introduction | What RNNs and LSTMs are — in plain language |
| 6.1 Why Sequences Are Different | Order matters · Hidden state concept · Business examples |
| 6.2 Vanilla RNN | The reading-a-sentence analogy · Unrolling · `nn.RNN` · Vanishing gradients |
| 6.3 LSTM | Gates as selective memory · Cell state · RNN vs LSTM comparison · `nn.LSTM` |
| 6.4 Business Application: CO₂ Forecasting | Mauna Loa dataset · z-score + week embedding · `LSTMWithWeekEmbed` · Train · Forecast |
| 6.5 Pipeline: Saving the Forecast Model | `ModelPipeline` for sequences · Save · Load · Forecast · Retrain |
| 6.6 ★ Bonus: Autoencoders | Encoder–bottleneck–decoder · Anomaly detection · CNN (image restoration) · LSTM (time-series) |

> **Dataset:** Mauna Loa CO₂ — built into `statsmodels`. No download, no account, no API key required.  
> **Model:** `LSTMWithWeekEmbed` — LSTM with a learned week-of-year embedding (defined in Section 6.4).

---
## Before We Begin: What Are RNNs and LSTMs?

Every architecture you have built so far treats each input as a standalone snapshot. Feed a customer record to an FFN (Chapter 4) and it produces a prediction without knowing whether that customer has been with you for one month or ten years. Feed a product image to a CNN (Chapter 5) and it detects defects without knowing whether the previous twenty images on the same line were also defective.

That is fine when inputs really are independent. But many of the most valuable datasets in business are not independent — they are **sequences**: ordered streams where the past directly shapes what comes next.

Think of reading a sentence. You do not treat each word in isolation. You carry meaning forward as you read — each word is understood in the context of everything that came before it. "The bank was steep" means something very different from "The bank was closed" — and you can only tell the difference because you remember the surrounding words.

**Recurrent Neural Networks (RNNs)** were designed to do exactly this: process sequences by maintaining a running **hidden state** — a compact summary of everything seen so far — that is updated at each step. Think of it as the network's short-term memory.

**Long Short-Term Memory networks (LSTMs)** are an improved version of RNNs that solve a critical weakness: vanilla RNNs tend to forget information from early in a sequence before they reach the end. LSTMs fix this by adding a carefully controlled long-term memory channel that can hold information across hundreds of steps.

By the end of this chapter you will have trained an LSTM to forecast atmospheric CO₂ concentration — a real scientific dataset with a strong trend and a beautiful yearly seasonal pattern — and watched it predict the future from 40 years of history.

> **What you already know that applies here:**  
> The training loop, loss functions, optimisers, and `ModelPipeline` from Chapters 3–5 are all used unchanged. The only new idea is *how the model processes its input* — one step at a time, with memory.

---
## Setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Chapter 6 — Setup
# Run this cell first. All packages except statsmodels are already on Colab;
# statsmodels installs in ~10 seconds.
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q statsmodels dill

import copy, datetime, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import statsmodels.api as sm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  |  device: {device}')
print('Setup complete.')

---
## Shared Training Helpers

These four helpers are identical to the versions introduced in Chapter 3 and reused in Chapters 4 and 5. They are copied here so this notebook is self-contained.

> **One addition:** `train_one_epoch` now calls `nn.utils.clip_grad_norm_` before the weight update. **Gradient clipping** is a standard best practice for RNNs and LSTMs — it prevents the rare but catastrophic case where gradients grow very large during backpropagation through long sequences. You will not notice it in normal training; it just quietly prevents instability.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Shared helpers — same contract as Chapters 3, 4, and 5.
# One addition: gradient clipping in train_one_epoch (best practice for RNNs).
# ─────────────────────────────────────────────────────────────────────────────

def train_one_epoch(model, loader, criterion, optimiser, device):
    model.train()
    total = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimiser.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # RNN best practice
        optimiser.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            total += criterion(model(X_batch), y_batch).item()
    return total / len(loader)


def run_training(model, train_loader, val_loader, criterion, optimiser,
                 num_epochs, device, scheduler=None, val_loss_scheduler=None,
                 early_stopping=None, verbose_every=5):
    """Standard training loop. Returns (train_losses, val_losses)."""
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        tl = train_one_epoch(model, train_loader, criterion, optimiser, device)
        vl = evaluate(model, val_loader, criterion, device)
        train_losses.append(tl)
        val_losses.append(vl)
        if scheduler:          scheduler.step()
        if val_loss_scheduler: val_loss_scheduler.step(vl)
        if verbose_every and (epoch % verbose_every == 0 or epoch == num_epochs - 1):
            print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
        if early_stopping and early_stopping.step(vl, model):
            print(f'  Early stopping at epoch {epoch}  (best val: {early_stopping.best_loss:.4f})')
            early_stopping.restore_best(model)
            break
    return train_losses, val_losses


class EarlyStopping:
    """Halt training when validation loss stops improving. Identical to Chapters 3–5."""
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best_loss = float('inf'); self.counter = 0; self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss; self.counter = 0
            self.best_weights = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
        return self.counter >= self.patience

    def restore_best(self, model):
        if self.best_weights: model.load_state_dict(self.best_weights)




def make_loaders(X, y, val_split=0.15, batch_size=64):
    """Split (X, y) tensors into train and val DataLoaders.
    Used in Sections 6.2 and 6.3 for sine-wave demonstration datasets.
    """
    n = len(X); split = int(n * (1 - val_split))
    tr = DataLoader(TensorDataset(X[:split], y[:split]), batch_size=batch_size, shuffle=True)
    va = DataLoader(TensorDataset(X[split:], y[split:]), batch_size=batch_size, shuffle=False)
    return tr, va

print('Helpers defined: train_one_epoch | evaluate | run_training | EarlyStopping | make_loaders')

---
# 6.1 Why Sequences Are Different

## Order matters

"Dog bites man" and "Man bites dog" contain exactly the same words. They are very different sentences. Order is meaning.

This is not a quirk of language. The same principle applies across all the sequential data that businesses care about:

| Data | Each step | Why order matters |
|------|-----------|-------------------|
| Daily sales | One day's revenue | Trend, seasonality, and momentum are temporal — yesterday's sales matter |
| Energy consumption | Hourly power reading | Morning and evening peaks repeat in predictable sequence |
| Customer clickstream | One page visit | The path taken so far determines the most likely next action |
| Sensor readings | One measurement | An anomaly is only visible as a deviation from recent history |

An FFN sees each input as a frozen snapshot. It has no way to carry information from one step to the next. To process a sequence of 60 readings, an FFN would have to receive all 60 at once as a single flat vector — which destroys the temporal relationships between them.

Recurrent networks were designed specifically to avoid this: they process sequences **one step at a time**, maintaining a **hidden state** that travels forward through the sequence like a traveller carrying notes from each stop.

## The hidden state — the central idea

Imagine reading a long paragraph by taking notes as you go. After each sentence you jot down a brief summary of everything you have read so far. When you reach the next sentence, you read it in the context of your notes — not just in isolation.

That is exactly what a recurrent network does. At every time step $t$:

1. It receives the **current input** $x_t$ (e.g. today's CO₂ reading)
2. It reads the **previous hidden state** $h_{t-1}$ (its notes so far)
3. It produces a **new hidden state** $h_t$ (updated notes)
4. Optionally, it produces an **output** $y_t$ (a prediction)

```
         x₁          x₂          x₃          x₄
         │           │           │           │
    ┌────▼────┐ h₁  ┌▼────────┐ h₂  ┌▼────────┐ h₃  ┌▼────────┐
    │  RNN    ├────►│  RNN    ├────►│  RNN    ├────►│  RNN    │
    │  cell   │     │  cell   │     │  cell   │     │  cell   │
    └────┬────┘     └────┬────┘     └────┬────┘     └────┬────┘
         │               │               │               │
         y₁              y₂              y₃              y₄
```

*Figure 6.1 — A recurrent network unrolled through four time steps. The same cell (same weights) is applied at every step. The hidden state $h_t$ carries information forward.*

> **Important:** The weights inside the RNN cell are **shared across all time steps**. The same transformation is applied at step 1 as at step 100. This is why an RNN can handle sequences of any length — there are no extra parameters for longer sequences.

### 📝 Exercise 6.1 — Sequence Problems in Your Domain

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.1
#
# Think of three sequential datasets from your own field or industry.
# For each one, fill in the three lines below.
#
# Example:
#   Sequence : Daily number of customer support tickets
#   Target   : Volume tomorrow (to plan staffing)
#   Why order: Monday spikes, weekends drop — the pattern only exists in time
#
# Problem 1:
#   Sequence : ?
#   Target   : ?
#   Why order: ?
#
# Problem 2:
#   Sequence : ?
#   Target   : ?
#   Why order: ?
#
# Problem 3:
#   Sequence : ?
#   Target   : ?
#   Why order: ?
#
# See the Answer Key at the end of this chapter.
# ─────────────────────────────────────────────────────────────────────────────

print('Exercise 6.1 complete.')

---
# 6.2 Vanilla RNN

## The reading-a-sentence analogy

A vanilla RNN works like a reader who keeps a very short notepad.

At each word (time step), the reader:
1. Looks at the current word
2. Glances at their current notepad summary
3. Erases the notepad and writes a new summary combining both

The problem: the notepad is small and gets **completely rewritten at every step**. By the time the reader reaches word 50, almost nothing from word 1 survives. The early context is washed out.

## The maths — one equation, applied repeatedly

The vanilla RNN update is a **single equation**:

$$h_t = \tanh(W_h \cdot h_{t-1} + W_x \cdot x_t + b)$$

The same equation is applied at **every time step** using the same weights. To see how memory builds up step by step, write it out explicitly:

$$h_1 = \tanh(W_h \cdot h_0 + W_x \cdot x_1 + b)$$
*Step 1 — blank slate: $h_0 = \mathbf{0}$, read $x_1$*

$$h_2 = \tanh(W_h \cdot h_1 + W_x \cdot x_2 + b)$$
*Step 2 — read $x_2$ in context of step 1*

$$h_3 = \tanh(W_h \cdot h_2 + W_x \cdot x_3 + b)$$
*Step 3 — read $x_3$ in context of steps 1–2*

$$h_4 = \tanh(W_h \cdot h_3 + W_x \cdot x_4 + b)$$
*Step 4 — read $x_4$ in context of steps 1–3*

Each $h_t$ depends on $h_{t-1}$, which depends on $h_{t-2}$, and so on back to $h_0$. In principle, $h_{50}$ contains information from all 50 previous steps. In practice, the chain of $\tanh$ squashings makes the very early context nearly impossible to recover — the vanishing gradient problem, explained in full at the end of this section.

> **Starting state $h_0$:** At step 1 there is no previous hidden state, so $h_0$ is simply a vector of zeros — a blank notepad.

In plain English: at each step, take the previous hidden state and the current input, mix them through two learned weight matrices, add a bias, and squash through $\tanh$ to stay in (−1, 1).

When the RNN needs to produce an output at each step (e.g. a prediction), a second equation maps the hidden state to the output, optionally followed by an activation function $\phi$:

$$y_t = \phi(W_y \cdot h_t + b_y)$$

The choice of activation depends on the task:

| Task | Activation $\phi$ | Reason |
|------|-------------------|--------|
| Regression (e.g. CO₂ ppm) | None (identity) | Output is unbounded — any real number |
| Binary classification | Sigmoid | Maps to probability (0–1) |
| Multi-class classification | Softmax | Maps to probability distribution |

For **sequence-to-one** forecasting (predict one value at the end), we only apply this equation once — at the final step $T$:

$$y = \phi(W_y \cdot h_T + b_y)$$

> **In PyTorch's `nn.RNN`:** the output equation is not built in. We add it as a `nn.Linear` layer on top — exactly what `self.fc = nn.Linear(hidden_size, 1)` does in the model below. For regression there is no activation after `fc`; for classification we would add `nn.Sigmoid()` or `nn.Softmax()`.

**Table 6.1 — The parts of an RNN**

| Symbol | Name | What it is |
|--------|------|------------|
| $x_t$ | Input at step $t$ | One element of the sequence (one week's CO₂ reading, one word token) |
| $h_{t-1}$ | Previous hidden state | The network's memory of everything before step $t$ |
| $h_t$ | New hidden state | Updated memory after reading $x_t$ |
| $W_x$ | Input weight matrix | How much to weight the current input |
| $W_h$ | Recurrent weight matrix | How much to weight the previous memory |
| $W_y$ | Output weight matrix | Maps the hidden state to the output (implemented as `nn.Linear`) |
| $b_y$ | Output bias | Additive offset in the output equation |
| $\phi$ | Output activation | Identity for regression; sigmoid/softmax for classification |
| $\tanh$ | Hidden activation | Keeps hidden state values bounded between −1 and 1 |

## The vanishing gradient problem — explained without maths

Here is the key weakness of the vanilla RNN, explained as a game of telephone.

Imagine 50 people standing in a line. A message is whispered from person 1 to person 2, then person 2 passes it to person 3, and so on. Each person slightly garbles the message before passing it on.

By the time the message reaches person 50, the original message from person 1 is almost unrecognisable.

This is exactly what happens during backpropagation in a vanilla RNN. When the network learns, it sends error signals *backwards* through the time steps — from step 50 back to step 1. But at each step, the gradient is multiplied by the recurrent weight matrix $W_h$. If the values in $W_h$ are slightly less than 1, multiplying 50 times produces a number close to zero. The gradient **vanishes** before it reaches the early steps.

The result: **the network cannot learn from context that happened more than ~10–20 steps ago**. On short sequences this is fine. On longer ones — a year of weekly data, a multi-sentence document — the early signal is lost.

To see why, remember that **the same weight matrix $W_h$ is reused at every time step**. During backpropagation, the gradient at step 1 is obtained by multiplying $W_h$ by itself once for every step we travel backward — so after 50 steps, the gradient has been multiplied by $W_h$ fifty times. If the values in $W_h$ are slightly less than 1 (which is typical after initialisation), this repeated multiplication makes the gradient exponentially smaller with each step. By the time the signal reaches step 1, it is so close to zero that the weights responsible for the earliest inputs receive no meaningful update — the model simply stops learning from that far back.

> **Why does this matter for forecasting?**  
> CO₂ has a yearly cycle. To forecast next week's reading, the model needs to remember what happened 52 weeks ago. A vanilla RNN simply cannot hold onto information that long. Section 6.3 shows how LSTMs solve this.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.2a — Vanilla RNN model
#
# nn.RNN(batch_first=True) expects input shape: (batch, seq_len, input_size)
# It returns:
#   output : (batch, seq_len, hidden_size) — hidden state at every step
#   h_n    : (num_layers, batch, hidden_size) — hidden state at the LAST step
#
# For sequence-to-one forecasting we only need the final hidden state h_n.
# ─────────────────────────────────────────────────────────────────────────────

class VanillaRNN(nn.Module):
    """Single-layer vanilla RNN — predicts one value from a sequence."""
    def __init__(self, input_size=1, hidden_size=32, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity='tanh',
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        _, h_n = self.rnn(x)      # h_n: (num_layers, batch, hidden_size)
        return self.fc(h_n[-1])   # use last layer's final hidden state


rnn_demo = VanillaRNN(input_size=1, hidden_size=32)
n_params = sum(p.numel() for p in rnn_demo.parameters())
print('VanillaRNN architecture:')
print(rnn_demo)
print(f'\nTrainable parameters: {n_params:,}')
print()

# Quick shape check — does it run?
x_demo = torch.randn(4, 10, 1)   # batch=4, seq_len=10, features=1
y_demo = rnn_demo(x_demo)
print(f'Input shape : {tuple(x_demo.shape)}')
print(f'Output shape: {tuple(y_demo.shape)}  (one prediction per sample — correct)')

**VanillaRNN parameter breakdown** (`input_size=1`, `hidden_size=32`)

> **One hidden layer — three weight matrices.** A single-layer RNN has only one hidden layer, but it uses three weight matrices. $W_x$ connects the input to the hidden state. $W_h$ connects the hidden state *at the previous time step* back to itself at the current time step — this is the **recurrent connection**, not a second layer. $W_y$ (implemented as `fc`) maps the final hidden state to the output.


> **Convention:** weight shapes are written as `(output_size × input_size)` — rows = destination neurons, columns = source neurons.

| Component | Shape (out × in)&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Weights | Bias | Total |
|:----------|:---------------------------------------------|--------:|-----:|------:|
| Input → Hidden ($W_x$) | $(32 \times 1)$ — 32 hidden units, 1 input | 32 | 32 | **64** |
| Previous hidden → Current hidden ($W_h$) | $(32 \times 32)$ — 32 hidden units, 32 prev hidden | 1,024 | 32 | **1,056** |
| Hidden → Output FC ($W_y$) | $(1 \times 32)$ — 1 output, 32 hidden inputs | 32 | 1 | **33** |
| **Total** | | | | **1,153** |

> $W_h$ is the **recurrent weight matrix** — it connects $h_{t-1}$ (last step's hidden state) to $h_t$ (this step's hidden state) within the same single layer. It is **not** a connection between two hidden layers. With `hidden_size=32`, $W_h$ is $32 \times 32 = 1{,}024$ values — this is what gets repeatedly multiplied during backpropagation through time and causes the vanishing gradient.


---
# 6.3 LSTM

## The core idea: two memory channels

The **Long Short-Term Memory (LSTM)** network, introduced by Hochreiter and Schmidhuber in 1997, solves the vanishing gradient problem with one elegant idea: instead of a single hidden state that gets completely overwritten at every step, the LSTM keeps **two separate memory channels**.

| Channel | Name | Role |
|---------|------|------|
| $h_t$ | **Hidden state** | Short-term output — passed to the next step and to the prediction layer |
| $c_t$ | **Cell state** | Long-term memory — travels through the sequence with only small, controlled changes |

Think of it as the difference between your working memory (what you are actively thinking about right now) and your long-term memory (knowledge you have accumulated over years). The hidden state is working memory; the cell state is long-term memory.

## The three gates — selective memory

The cell state is not just a second hidden state. It is controlled by **four learned computations** — three gates and one candidate — that work together to decide what information flows in, out, or gets erased.

| Computation | Type | Activation | Role |
|-------------|------|-----------|------|
| Forget gate $f_t$ | Gate | Sigmoid → (0,1) | How much of the old cell state to keep |
| Input gate $i_t$ | Gate | Sigmoid → (0,1) | How much of the new candidate to write |
| Cell candidate $\tilde{c}_t$ | Candidate | Tanh → (−1,1) | What new content to propose for storage |
| Output gate $o_t$ | Gate | Sigmoid → (0,1) | How much of the cell state to expose as $h_t$ |

The three **gates** ($f_t$, $i_t$, $o_t$) use sigmoid and act as smooth on/off switches — values near 0 mean "block", near 1 mean "pass through". The **candidate** $\tilde{c}_t$ uses tanh and proposes new content. The input gate then decides how much of that proposal to actually store. This is why people sometimes say "three gates" (referring only to the sigmoid gates) and sometimes "four computations" (including the candidate) — both descriptions are used in the literature. In PyTorch and in our parameter count, all four have their own weight matrices, which is exactly why an LSTM has **~4× more parameters** than a vanilla RNN of the same hidden size.

```
 ┌─────────────────────────────────────────────────────────────────────────┐
 │                        LSTM Cell at step t                              │
 │                                                                         │
 │  Inputs:  x_t  (current value)   h_{t-1}  (short-term memory)          │
 │           c_{t-1}  (long-term memory — the "highway")                   │
 │                                                                         │
 │  ══════════════ LONG-TERM MEMORY HIGHWAY (cell state) ════════════════  │
 │                                                                         │
 │  c_{t-1} ─► [✕ FORGET gate] ─────────────► [+ ] ────────────► c_t     │
 │                    │                         ↑                          │
 │                "Erase old           [✕ INPUT gate] ─► [tanh candidate]  │
 │                 memories"                   │                           │
 │                                      "Write new info"                   │
 │                                                                         │
 │  ══════════════ SHORT-TERM OUTPUT (hidden state) ══════════════════════  │
 │                                                                         │
 │  c_t ─► [tanh] ─► [✕ OUTPUT gate] ────────────────────────────► h_t   │
 │                           │                                             │
 │                    "What to expose"                                     │
 │                                                                         │
 │  ⚠  Every gate (forget, input, output) takes BOTH x_t AND h_{t-1}.     │
 └─────────────────────────────────────────────────────────────────────────┘
```

*Figure 6.3 — LSTM cell. The cell state $c_t$ (top highway) updates additively — this is why gradients survive long sequences. The hidden state $h_t$ (bottom) is what the prediction head sees.*
### Full equations — walking through a concrete example

Rather than presenting the equations in the abstract, let us trace them with a tiny example first. This makes the concatenation, the gates, and the two memory channels all concrete before we generalise.

**Setup:** `input_size = 1`, `hidden_size = 2`. One sequence of two steps.
- Step $t$: input $x_t = 0.5$, previous hidden state $h_{t-1} = [0.3, -0.1]$, previous cell state $c_{t-1} = [0.2, 0.0]$

**Step 1 — Concatenate $h_{t-1}$ and $x_t$**

Every gate starts by concatenating the previous hidden state and the current input into a single vector:

$$[h_{t-1},\, x_t] = [0.3,\; -0.1,\; 0.5] \quad \text{(length = hidden\_size + input\_size = 3)}$$

This concatenated vector is what every gate operates on. **All four gates see it simultaneously.**

**Step 2 — Forget gate** (decide what fraction of $c_{t-1}$ to keep):

$$f_t = \sigma(W_f \cdot [h_{t-1},\, x_t] + b_f)$$

With learned weights this might produce, say, $f_t = [0.8,\; 0.6]$. These are the "keep fractions" — 80% of $c_{t-1}[0]$ and 60% of $c_{t-1}[1]$ will survive.

**Step 3 — Input gate + candidate** (decide what new information to write):

$$i_t = \sigma(W_i \cdot [h_{t-1},\, x_t] + b_i) \quad \rightarrow \text{e.g.}\; [0.4,\; 0.7]$$

$$\tilde{c}_t = \tanh(W_c \cdot [h_{t-1},\, x_t] + b_c) \quad \rightarrow \text{e.g.}\; [0.9,\; -0.5]$$

$\tilde{c}_t$ is the *proposed* new memory. $i_t$ decides how much of it to actually store.

**Step 4 — Cell state update** (the long-term memory highway):

$$c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$$

$$c_t = [0.8,\; 0.6] \odot [0.2,\; 0.0] + [0.4,\; 0.7] \odot [0.9,\; -0.5]$$

$$c_t = [0.16,\; 0.00] + [0.36,\; -0.35] = [0.52,\; -0.35]$$

Notice: this is **addition**, not matrix multiplication. Gradients flowing back through $c_t$ pass through addition and element-wise multiplication — they do not shrink through repeated matrix multiplies.

**Step 5 — Output gate** (decide what to surface as the hidden state):

$$o_t = \sigma(W_o \cdot [h_{t-1},\, x_t] + b_o) \quad \rightarrow \text{e.g.}\; [0.5,\; 0.8]$$

$$h_t = o_t \odot \tanh(c_t) = [0.5,\; 0.8] \odot \tanh([0.52,\; -0.35])$$

$$h_t = [0.5,\; 0.8] \odot [0.48,\; -0.34] = [0.24,\; -0.27]$$

This $h_t$ is passed to the next step as the new "short-term memory" and to the output layer as the prediction.

**Full gate summary**

| Gate | Formula&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Range | Role |
|:-----|:--------------------------------------------------|:-----:|:-----|
| Forget $f_t$ | $\sigma(W_f \cdot [h_{t-1},\ x_t] + b_f)$ | (0, 1) | How much of $c_{t-1}$ to keep |
| Input $i_t$ | $\sigma(W_i \cdot [h_{t-1},\ x_t] + b_i)$ | (0, 1) | How much new info to write |
| Candidate $\tilde{c}_t$ | $\tanh(W_c \cdot [h_{t-1},\ x_t] + b_c)$ | (−1, 1) | Proposed new memory |
| Cell $c_t$ | $f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ | Unbounded | Long-term memory (additive update) |
| Output $o_t$ | $\sigma(W_o \cdot [h_{t-1},\ x_t] + b_o)$ | (0, 1) | How much of $c_t$ to expose |
| Hidden $h_t$ | $o_t \odot \tanh(c_t)$ | (−1, 1) | Short-term output to next step |

> **Key insight:** A vanilla RNN has **one** transformation: $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$.  
> An LSTM has **four parallel transformations** — forget, input, candidate, output — each with its own weight matrices. This is why LSTM has ~4× more parameters than an RNN of the same hidden size.

## Why this solves the vanishing gradient

The vanilla RNN multiplies the hidden state by $W_h$ at every step. Multiply by a number slightly less than 1, fifty times, and you get essentially zero.

The cell state is updated **additively**: $c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$. Addition preserves gradient magnitude — it does not shrink it. When the forget gate is near 1 ("keep this memory"), gradients flow backward through the cell state across many steps without shrinking. The network can genuinely learn from context that is 52 steps (one year) or 200 steps ago.

## RNN vs LSTM — summary comparison

| Feature | Vanilla RNN | LSTM |
|---------|-------------|------|
| Memory channels | 1 (hidden state only) | 2 (hidden state + cell state) |
| Update mechanism | Overwrite (multiplicative) | Additive with gating |
| Long-range dependencies | Struggles beyond ~20 steps | Handles hundreds of steps |
| Parameters (same hidden size) | ~1× | ~4× (one set per gate) |
| Training stability | Can explode or vanish | Much more stable |
| PyTorch class | `nn.RNN` | `nn.LSTM` |
| When to use | Short sequences (<20 steps) | Sequences of any length |

> **Rule of thumb:** If your sequence is longer than 20 steps, use an LSTM. The extra parameters are worth it.

> **Three gates vs four computations — both phrasings are correct:**  
> You will see both in textbooks and papers. The distinction is whether the cell candidate $\tilde{c}_t$ is counted separately:
> - **"Three gates"** = forget $f_t$, input $i_t$, output $o_t$ (all sigmoid) — referring only to the gating mechanisms
> - **"Four computations"** = the above three plus the cell candidate $\tilde{c}_t$ (tanh) — counting all four weight-matrix operations
> In our parameter breakdown and in PyTorch's internal layout, all four have their own weight matrices. That is why an LSTM costs ~4× more than a vanilla RNN of the same `hidden_size`, not ~3×.

> **Coming up in Section 6.4:** The CO₂ forecasting application extends `LSTMNet` with a learned **week-of-year embedding** — each of the 52 weeks gets its own representation. This `LSTMWithWeekEmbed` is defined right before the training cell where it is first used.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3a — LSTM model
#
# nn.LSTM is a drop-in replacement for nn.RNN.
# It returns (output, (h_n, c_n)) — note the tuple.
# We unpack (h_n, c_n) and use only h_n for the prediction head.
# ─────────────────────────────────────────────────────────────────────────────

class LSTMNet(nn.Module):
    """LSTM for sequence-to-one regression or classification.
    Constructor kwargs match model_config for ModelPipeline compatibility.
    """
    def __init__(self, input_size=1, hidden_size=64, num_layers=2,
                 dropout=0.2, output_size=1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)      # h_n: (num_layers, batch, hidden_size)
        out = self.dropout(h_n[-1])     # last layer, final time step
        return self.fc(out)             # (batch, output_size)


lstm_demo = LSTMNet(input_size=1, hidden_size=64, num_layers=2, dropout=0.2)
n_rnn  = sum(p.numel() for p in VanillaRNN(hidden_size=64).parameters())
n_lstm = sum(p.numel() for p in lstm_demo.parameters())
print('LSTMNet architecture:')
print(lstm_demo)
print(f'\nVanillaRNN parameters (hidden=64): {n_rnn:,}')
print(f'LSTMNet   parameters (hidden=64): {n_lstm:,}')
print(f'LSTM has {n_lstm / n_rnn:.1f}× more parameters — 4 weight sets, one per gate.')

**LSTMNet parameter breakdown** (`input_size=1`, `hidden_size=64`, `num_layers=2`)

**Why ~4× more parameters than a vanilla RNN?**

A vanilla RNN applies **one transformation** at each step:

$$h_t = \tanh(W_x \cdot x_t + W_h \cdot h_{t-1} + b) \quad \text{— 1 set of weight matrices}$$

An LSTM applies **four parallel transformations** at each step — three sigmoid gates ($f_t$, $i_t$, $o_t$) and one tanh candidate ($\tilde{c}_t$) — each with its own full set of weight matrices:

| Computation | Type | Weight matrices needed |
|-------------|------|------------------------|
| Forget gate $f_t$ | Sigmoid gate | $W_f^{(x)}$ and $W_f^{(h)}$ |
| Input gate $i_t$ | Sigmoid gate | $W_i^{(x)}$ and $W_i^{(h)}$ |
| Cell candidate $\tilde{c}_t$ | Tanh (not a gate) | $W_c^{(x)}$ and $W_c^{(h)}$ |
| Output gate $o_t$ | Sigmoid gate | $W_o^{(x)}$ and $W_o^{(h)}$ |

Four sets of weight matrices instead of one → approximately 4× the parameters. PyTorch stores all four sets stacked together in `weight_ih` and `weight_hh`, which is why each has shape `(4 × hidden_size, ...)` in the breakdown below.

**Layer 1** — `input_size = 1 → hidden_size = 64`

| Component&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Calculation&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Params |
|:------|:------|------:|
| Input → Hidden weights (4 gates)  | $4 \times (64 \times 1)$  | 256 |
| Hidden → Hidden weights (4 gates) | $4 \times (64 \times 64)$ | 16,384 |
| Input bias (4 gates)              | $4 \times 64$              | 256 |
| Hidden bias (4 gates)             | $4 \times 64$              | 256 |
| **Layer 1 Total**                 |                             | **17,152** |

**Layer 2** — `input_size = 64 → hidden_size = 64` (previous layer's output feeds in)

| Component&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Calculation&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Params |
|:------|:------|------:|
| Input → Hidden weights (4 gates)  | $4 \times (64 \times 64)$ | 16,384 |
| Hidden → Hidden weights (4 gates) | $4 \times (64 \times 64)$ | 16,384 |
| Input bias (4 gates)              | $4 \times 64$              | 256 |
| Hidden bias (4 gates)             | $4 \times 64$              | 256 |
| **Layer 2 Total**                 |                             | **33,280** |

**Fully Connected output layer**

| Component&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Calculation&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp; | Params |
|:------|:------|------:|
| Weights      | $64 \times 1$ | 64 |
| Bias         | $1$            | 1  |
| **FC Total** |                | **65** |

**Grand Total**

| Part | Params |
|------|-------:|
| LSTM Layer 1 | 17,152 |
| LSTM Layer 2 | 33,280 |
| FC Layer | 65 |
| **Total** | **50,497** |

> The `n_params` printed by the code above should equal **50,497**. Verify it matches!


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.3 — Verify the worked example with PyTorch
#
# The concept section above walked through a 2-step LSTM by hand.
# This cell verifies the same computation with nn.LSTM(input=1, hidden=2).
# We print gate values and cell/hidden states so you can match them
# to the step-by-step derivation above.
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(0)
tiny = nn.LSTM(input_size=1, hidden_size=2, num_layers=1, batch_first=True)

# Two time steps: x_0=0.5, x_1=1.2
x_seq = torch.tensor([[[0.5], [1.2]]])   # (batch=1, seq=2, input=1)
h0    = torch.zeros(1, 1, 2)
c0    = torch.zeros(1, 1, 2)

with torch.no_grad():
    output, (h_n, c_n) = tiny(x_seq, (h0, c0))

print('nn.LSTM(input=1, hidden=2) — two steps:')
print(f'  x_0=0.5, h_{{t-1}}=zeros, c_{{t-1}}=zeros')
print(f'  h_1 (after step 0): {output[0,0].tolist()}')
print(f'  h_2 (after step 1): {output[0,1].tolist()}')
print(f'  c_final           : {c_n[0,0].tolist()}')
print()

# Manually compute gate values at step 0 to match the walkthrough
hs = 2
W_ih = tiny.weight_ih_l0  # (4*hs, 1)
W_hh = tiny.weight_hh_l0  # (4*hs, hs)
b_ih = tiny.bias_ih_l0
b_hh = tiny.bias_hh_l0

x_t    = torch.tensor([[0.5]])     # (1, 1)
h_prev = torch.zeros(1, hs)        # (1, 2)
c_prev = torch.zeros(1, hs)        # (1, 2)

pre = x_t @ W_ih.T + b_ih + h_prev @ W_hh.T + b_hh  # (1, 8)
i_g = torch.sigmoid(pre[:, 0*hs : 1*hs])   # input gate
f_g = torch.sigmoid(pre[:, 1*hs : 2*hs])   # forget gate
g_g = torch.tanh(   pre[:, 2*hs : 3*hs])   # candidate
o_g = torch.sigmoid(pre[:, 3*hs : 4*hs])   # output gate
c_1 = f_g * c_prev + i_g * g_g
h_1 = o_g * torch.tanh(c_1)

fmt = lambda t: [round(v, 4) for v in t[0].tolist()]
print('Step 0 gate values (matches the walkthrough in the concept cell):')
print(f'  Forget gate  f_t : {fmt(f_g)}  ← how much of c_prev to keep')
print(f'  Input gate   i_t : {fmt(i_g)}  ← how much new info to write')
print(f'  Candidate   g_t  : {fmt(g_g)}  ← proposed new memory')
print(f'  Output gate  o_t : {fmt(o_g)}  ← how much to expose')
print(f'  Cell state   c_1 : {fmt(c_1)}  ← f*c_prev + i*g  (additive!)')
print(f'  Hidden state h_1 : {fmt(h_1)}  ← o * tanh(c_1)')
print(f'  Matches nn.LSTM  : {fmt(output[:, 0, :])}  ✓')


### 📝 Exercise 6.3 — Replace RNN with LSTM

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.3
#
# Task: Train an LSTMNet on the SHORT sequence dataset (seq_len=10).
#       Plot its val loss alongside the RNN short-sequence result.
#
# After running, answer these questions as comments:
#   Q1. On short sequences, is the LSTM clearly better than the RNN?
#       What does this tell you about when to use each architecture?
#   Q2. The LSTM has ~4× more parameters than the RNN for the same hidden size.
#       Is that extra cost always justified?
# ─────────────────────────────────────────────────────────────────────────────

torch.manual_seed(SEED)
# X_short, y_short = make_sine_dataset(SHORT)
# tr_short, va_short = make_loaders(X_short, y_short)
# lstm_short = LSTMNet(input_size=1, hidden_size=32, num_layers=1, dropout=0.0).to(device)
# _, lstm_short_val = run_training(
#     lstm_short, tr_short, va_short, nn.MSELoss(),
#     optim.Adam(lstm_short.parameters(), lr=1e-3),
#     num_epochs=40, device=device, verbose_every=None)
#
# fig, ax = plt.subplots(figsize=(9, 4))
# ax.plot(rnn_results[f'RNN  seq={SHORT}'], color='#27ae60', lw=2.0, label=f'RNN  seq={SHORT}')
# ax.plot(lstm_short_val,                   color='#2980b9', lw=2.0, label=f'LSTM seq={SHORT}')
# ax.set_xlabel('Epoch'); ax.set_ylabel('Validation MSE'); ax.legend()
# ax.set_title('Exercise 6.3 — RNN vs LSTM on short sequences')
# plt.tight_layout(); plt.show()
#
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.3 complete.')

---
# 6.4 Business Application: CO₂ Forecasting

## The business problem

Atmospheric CO₂ concentration is one of the most consequential time series in the world. Measured weekly at the Mauna Loa Observatory in Hawaii since 1958, it captures humanity's carbon footprint in a single number. Every organisation with a climate risk strategy, carbon credit programme, or net-zero commitment needs accurate CO₂ forecasts.

It is also a perfect teaching dataset, because its structure is unusually clean and highly learnable:

- **A strong upward trend** — concentration has risen from 316 ppm in 1958 to 372 ppm by 2001, driven by fossil fuel emissions (~0.9 ppm per year on average)
- **A precise yearly cycle** — CO₂ dips each northern-hemisphere summer as vegetation absorbs it, then rises each winter as plant activity falls (the "Keeling Curve" sawtooth)

A well-trained LSTM should learn both patterns and produce visibly accurate forecasts. If your model's predictions track the seasonal sawtooth into the future, you have built something real.

## The dataset

The data is built directly into `statsmodels` — one function call, no download, no account.

| Property | Value |
|----------|-------|
| Source | Scripps Institution of Oceanography / NOAA |
| Frequency | Weekly |
| Date range | 30 March 1958 – 30 December 2001 |
| Total observations | 2,284 weekly readings |
| Target variable | CO₂ concentration (parts per million, ppm) |
| Licence | Public domain |

In [ ]:
co2_raw = sm.datasets.co2.load_pandas().data
co2_raw = co2_raw.resample('W').mean().ffill()

daily_full = co2_raw.reset_index().rename(columns={'index': 'date', 'co2': 'co2_ppm'})

YEARS_TO_USE = 10
cutoff       = daily_full.date.max() - pd.DateOffset(years=YEARS_TO_USE)
daily        = daily_full[daily_full.date >= cutoff].reset_index(drop=True)

# Integer week index 0-51 — used by nn.Embedding in the model
daily['week_idx'] = ((daily.date.dt.dayofyear / 7).astype(int) % 52).astype(np.int64)

print(f'Subset   : {daily.date.min().date()} to {daily.date.max().date()}  ({len(daily):,} weeks)')
print(f'CO2 range: {daily.co2_ppm.min():.1f} – {daily.co2_ppm.max():.1f} ppm')
print(f'week_idx : {daily.week_idx.min()} – {daily.week_idx.max()}  (integer, 0=week1 … 51=week52)')


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
fig.patch.set_facecolor('white')

axes[0].set_facecolor('white')
axes[0].plot(daily.date, daily.co2_ppm, color='#2980b9', lw=1.0, alpha=0.9)
axes[0].set_title(f'(a) CO2 — Last {YEARS_TO_USE} Years  '
                  f'({daily.date.min().year}–{daily.date.max().year})',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[0].set_ylabel('CO2 (ppm)', fontsize=10)

zoom = daily[daily.date.dt.year >= daily.date.dt.year.max() - 3]
axes[1].set_facecolor('white')
axes[1].plot(zoom.date, zoom.co2_ppm, color='#e74c3c', lw=2.0)
axes[1].set_title('(b) Zoom: last 3 years — yearly sawtooth clearly visible',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[1].set_ylabel('CO2 (ppm)', fontsize=10)

fig.suptitle('Figure 6.5 — Mauna Loa CO2: upward trend + yearly seasonality',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()


In [ ]:
SEQ_LEN  = 104
VAL_FRAC = 0.20

co2_raw  = daily.co2_ppm.values.astype(np.float32)
CO2_MEAN = float(co2_raw.mean())
CO2_STD  = float(co2_raw.std())
co2_norm = (co2_raw - CO2_MEAN) / CO2_STD

# Integer week index from the column created in the load cell
week_idx = daily['week_idx'].values.astype(np.int64)
targets  = co2_norm

print(f'CO2_MEAN={CO2_MEAN:.2f}  CO2_STD={CO2_STD:.2f}')
print(f'co2_norm: {co2_norm.min():.2f} to {co2_norm.max():.2f}')
print(f'week_idx: {week_idx.min()} to {week_idx.max()}')
print()


def make_windows(co2_arr, wk_arr, tgt_arr, seq_len):
    co2_list, wk_list, y_list = [], [], []
    for i in range(len(co2_arr) - seq_len):
        co2_list.append(co2_arr[i : i + seq_len])
        wk_list.append(wk_arr[i  : i + seq_len])
        y_list.append(tgt_arr[i + seq_len])
    X_co2  = torch.tensor(np.array(co2_list), dtype=torch.float32).unsqueeze(-1)
    X_week = torch.tensor(np.array(wk_list),  dtype=torch.int64)
    y      = torch.tensor(np.array(y_list),   dtype=torch.float32).unsqueeze(-1)
    return X_co2, X_week, y


X_co2_all, X_week_all, y_all = make_windows(co2_norm, week_idx, targets, SEQ_LEN)

split    = int(len(X_co2_all) * (1 - VAL_FRAC))
tr_split = split
X_co2_tr  = X_co2_all[:split];  X_week_tr = X_week_all[:split];  y_tr = y_all[:split]
X_co2_va  = X_co2_all[split:];  X_week_va = X_week_all[split:];  y_va = y_all[split:]

co2_train_loader = DataLoader(TensorDataset(X_co2_tr, X_week_tr, y_tr), batch_size=32, shuffle=True)
co2_val_loader   = DataLoader(TensorDataset(X_co2_va, X_week_va, y_va), batch_size=32, shuffle=False)

print(f'Train windows: {len(X_co2_tr):,}  Val windows: {len(X_co2_va):,}')
print(f'X_co2 : {tuple(X_co2_tr.shape)}  X_week: {tuple(X_week_tr.shape)}  y: {tuple(y_tr.shape)}')


## A New Idea: Embeddings

Before building the model we need to introduce one new concept: **embeddings**. This is the first time they appear in this book, and they will become central again in Chapter 7 when we study Large Language Models.

### The problem with encoding categories as numbers

We have 52 distinct weeks in a year. The simplest way to tell the model which week it is would be a single integer: week 1 = 1, week 26 = 26, week 52 = 52. But this creates a problem. The model will interpret these numbers as having *magnitude* — week 52 appears to be 52 times more important than week 1, and the distance between week 1 and week 52 appears to be huge, even though in the calendar they are adjacent (the last week of the year is immediately followed by the first).

A scalar between 0 and 1 (e.g. `week / 51`) avoids the magnitude issue but creates another: weeks 25 and 26 differ by just 0.02, yet one is before the summer CO₂ minimum and one is after — behaviourally very different. The model must learn that a tiny numerical difference encodes a large behavioural one.

### What an embedding does

An **embedding** maps each category to a **learnable dense vector**. Instead of representing week 26 as the scalar `0.49`, the model represents it as a vector of, say, 8 numbers — all of which are learned end-to-end during training.

```
week_index  →  nn.Embedding(52, 8)  →  8-dimensional vector

week  1  →  [ 0.23, -0.71,  0.14,  0.55, -0.32,  0.08,  0.61, -0.19]  (winter)
week 26  →  [-0.44,  0.62, -0.83,  0.11,  0.71, -0.55, -0.12,  0.38]  (summer min)
week 52  →  [ 0.21, -0.68,  0.17,  0.52, -0.28,  0.11,  0.59, -0.22]  (near week 1)
```

The vectors are initialised randomly and updated by backpropagation like any other weight. The model learns to place weeks that *behave similarly* close together in vector space. After training, weeks near the summer minimum will have similar vectors; weeks near the winter maximum will have similar vectors; weeks 1 and 52 (adjacent in the calendar) will be close.

**Table 6.4 — Three ways to encode a categorical week index**

| Property | Raw integer (1–52) | Divided scalar (0–1) | **Embedding (8-dim)** |
|----------|--------------------|---------------------|----------------------|
| Represents each week as | One integer | One float | **Eight learned floats** |
| Week 1 and week 52 distance | 51 (huge — wrong) | 1.0 (large — wrong) | **~0.03 (correctly close)** |
| Week 1 and week 26 distance | 25 (large — wrong) | 0.49 (OK) | **~1.8 (correctly distant)** |
| Learns seasonal structure | No | No | **Yes — from data** |
| Parameters added | 0 | 0 | **52 × 8 = 416** |

> The raw integer and divided scalar columns show *why* previous approaches fail: they impose a false linear ordering on categories that have a circular, non-linear relationship.

### PyTorch syntax

```python
self.week_embed = nn.Embedding(num_embeddings=52, embedding_dim=8)
```

- `num_embeddings=52` — the vocabulary size (one row per week)
- `embedding_dim=8` — the length of each week's vector

At runtime, pass an integer tensor of week indices and get back a float tensor of vectors:

```python
week_idx = torch.tensor([0, 25, 51])          # three week indices
emb      = self.week_embed(week_idx)           # shape (3, 8)
```

> **The same idea in Chapter 7:** Large language models represent each word token as an embedding vector. The `nn.Embedding` layer you see here is *exactly* the same mechanism — just with 50,000 word tokens instead of 52 weeks, and 768 or 4096 dimensions instead of 8.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# LSTMWithWeekEmbed — LSTM with a learned week-of-year embedding
#
# Why an embedding instead of a scalar?
#   A scalar week index (e.g. 0.5 for week 26) forces the model to learn
#   that small numerical differences encode large behavioural differences.
#   Week 25 and week 27 differ by 0.04 — yet one is before the summer
#   minimum and one is after. An embedding gives each of the 52 weeks its
#   own learned representation. The model sees a rich 8-dimensional vector
#   for 'week 26' rather than a near-identical scalar to week 25.
#
# Architecture:
#   Input per step: CO2 z-score (1) + week embedding (EMBED_DIM) → EMBED_DIM+1
#   Then: standard 2-layer LSTM → Linear → 1 output
# ─────────────────────────────────────────────────────────────────────────────

EMBED_DIM = 8   # each of the 52 weeks gets an 8-dim learned vector
N_WEEKS   = 52  # number of distinct week indices (0–51)

class LSTMWithWeekEmbed(nn.Module):
    """LSTM forecaster with a learned week-of-year embedding.

    Inputs per time step:
        co2_z    : (batch, seq_len, 1)    — z-scored CO2
        week_idx : (batch, seq_len)       — integer week index 0-51

    The week embedding is concatenated to co2_z before the LSTM,
    giving an effective input_size of 1 + EMBED_DIM.
    """
    def __init__(self, hidden_size=128, num_layers=2, dropout=0.2,
                 n_weeks=N_WEEKS, embed_dim=EMBED_DIM):
        super().__init__()
        self.week_embed = nn.Embedding(n_weeks, embed_dim)
        self.lstm = nn.LSTM(
            input_size  = 1 + embed_dim,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_size, 1)

    def forward(self, co2_z, week_idx):
        # co2_z    : (batch, seq_len, 1)
        # week_idx : (batch, seq_len)  integer
        emb  = self.week_embed(week_idx)        # (batch, seq_len, embed_dim)
        x    = torch.cat([co2_z, emb], dim=-1)  # (batch, seq_len, 1+embed_dim)
        _, (h_n, _) = self.lstm(x)
        return self.fc(self.dropout(h_n[-1]))   # (batch, 1)


# Quick shape check
torch.manual_seed(SEED)
_m   = LSTMWithWeekEmbed().to(device)
_co2 = torch.randn(4, 104, 1)
_wk  = torch.randint(0, 52, (4, 104))
_out = _m(_co2.to(device), _wk.to(device))
print(f'LSTMWithWeekEmbed')
print(f'  Parameters  : {sum(p.numel() for p in _m.parameters()):,}')
print(f'  Input shapes: co2_z {tuple(_co2.shape)}  week_idx {tuple(_wk.shape)}')
print(f'  Output shape: {tuple(_out.shape)}')
del _m, _co2, _wk, _out


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4d — Train the CO2 forecast LSTM with week embedding
# ─────────────────────────────────────────────────────────────────────────────

CO2_CONFIG = dict(
    hidden_size = 128,
    num_layers  = 2,
    dropout     = 0.2,
    n_weeks     = N_WEEKS,
    embed_dim   = EMBED_DIM,
)

torch.manual_seed(SEED)
co2_model = LSTMWithWeekEmbed(**CO2_CONFIG).to(device)

co2_crit = nn.MSELoss()
co2_opt  = optim.Adam(co2_model.parameters(), lr=1e-3)
co2_sched = optim.lr_scheduler.CosineAnnealingLR(co2_opt, T_max=100, eta_min=1e-5)
co2_es   = EarlyStopping(patience=15)

print(f'Parameters: {sum(p.numel() for p in co2_model.parameters()):,}')
print('Training (up to 100 epochs):')
print()

# Custom training loop: loader yields (X_co2, X_week, y) — 3 tensors
def train_embed(model, loader, criterion, optimiser, device):
    model.train(); total = 0.0
    for co2_b, wk_b, y_b in loader:
        co2_b, wk_b, y_b = co2_b.to(device), wk_b.to(device), y_b.to(device)
        optimiser.zero_grad()
        loss = criterion(model(co2_b, wk_b), y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
        total += loss.item()
    return total / len(loader)


def eval_embed(model, loader, criterion, device):
    model.eval(); total = 0.0
    with torch.no_grad():
        for co2_b, wk_b, y_b in loader:
            co2_b, wk_b, y_b = co2_b.to(device), wk_b.to(device), y_b.to(device)
            total += criterion(model(co2_b, wk_b), y_b).item()
    return total / len(loader)


co2_train_hist, co2_val_hist = [], []
for epoch in range(100):
    tl = train_embed(co2_model, co2_train_loader, co2_crit, co2_opt, device)
    vl = eval_embed(co2_model,  co2_val_loader,   co2_crit, device)
    co2_train_hist.append(tl)
    co2_val_hist.append(vl)
    co2_sched.step()
    if epoch % 10 == 0 or epoch == 99:
        print(f'  Epoch {epoch:3d}: train={tl:.4f}  val={vl:.4f}')
    if co2_es.step(vl, co2_model):
        print(f'  Early stopping at epoch {epoch}')
        co2_es.restore_best(co2_model)
        break

print('\nTraining complete.')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.4e — Evaluate: MAE, MAPE, forecast vs actual
# ─────────────────────────────────────────────────────────────────────────────

co2_model.eval()
preds_z, actuals_z = [], []
with torch.no_grad():
    for co2_b, wk_b, y_b in co2_val_loader:
        out = co2_model(co2_b.to(device), wk_b.to(device)).cpu().numpy()
        preds_z.append(out)
        actuals_z.append(y_b.numpy())

preds_z   = np.concatenate(preds_z).flatten()
actuals_z = np.concatenate(actuals_z).flatten()

# Invert z-score
preds   = preds_z   * CO2_STD + CO2_MEAN
actuals = actuals_z * CO2_STD + CO2_MEAN

mae  = np.mean(np.abs(preds - actuals))
mape = np.mean(np.abs((preds - actuals) / actuals)) * 100
print(f'Validation MAE  : {mae:.3f} ppm')
print(f'Validation MAPE : {mape:.3f}%')

target_start = tr_split + SEQ_LEN
target_end   = min(target_start + len(actuals), len(daily))
val_dates    = daily.date.values[target_start : target_end]
n_show       = len(val_dates)
preds        = preds[:n_show]
actuals      = actuals[:n_show]

fig, axes = plt.subplots(2, 1, figsize=(13, 8))
fig.patch.set_facecolor('white')

axes[0].set_facecolor('white')
axes[0].plot(val_dates, actuals, color='#1F3864', lw=1.5, label='Actual CO2')
axes[0].plot(val_dates, preds,   color='#e74c3c', lw=1.5, label='LSTM forecast', alpha=0.85)
axes[0].set_title(f'(a) CO2 Forecast vs Actual  (MAE: {mae:.2f} ppm  MAPE: {mape:.2f}%)',
                  fontsize=11, fontweight='bold', color='#1F3864')
axes[0].set_ylabel('CO2 (ppm)', fontsize=10)
axes[0].legend(fontsize=9)

axes[1].set_facecolor('white')
ep = range(1, len(co2_train_hist) + 1)
axes[1].plot(ep, co2_train_hist, color='#2980b9', lw=2.0, label='Train MSE')
axes[1].plot(ep, co2_val_hist,   color='#e67e22', lw=2.0, label='Val MSE', linestyle='--')
axes[1].set_title('(b) Training Loss Curves', fontsize=11, fontweight='bold', color='#1F3864')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE Loss (z-score²)')
axes[1].legend(fontsize=9)

fig.suptitle('Figure 6.6 — CO2 Forecast LSTM  (z-scored CO2 + week embedding)',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()


### 📝 Exercise 6.4 — End-to-End CO₂ Forecasting

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.4
#
# Task A: Change SEQ_LEN to 26 (half a year).
#         Retrain the same LSTM. Does MAE improve or worsen?
#         Think about why: what seasonal context does a 26-week window miss?
#
# Task B: Change hidden_size to 32 (smaller model).
#         How does MAE change? Is the smaller model still acceptable?
#
# Task C: Answer as comments:
#   Q1. We used a temporal split (last 20% = validation).
#       Why would a random split be wrong for time series?
#   Q2. MAPE divides by the actual value. Why is it useful for CO₂ forecasting
#       even though CO₂ values are never near zero?
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# SEQ_LEN_A = 26
# X_a, y_a = make_windows(scaled, SEQ_LEN_A)
# ...  (same train/val split and training loop)

# Task B
# CO2_CONFIG_B = {**CO2_CONFIG, 'hidden_size': 32}
# ...

# Task C
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.4 complete.')

---
# 6.5 Pipeline: Saving and Deploying the Forecast Model

Every chapter from Chapter 3 onward closes with the same professional habit: wrap the trained model in a `ModelPipeline` and save it. For sequence models, there is one important addition — the **window configuration** (how many past steps the model expects) must be saved alongside the weights.

If you load a saved LSTM and feed it 30 steps when it was trained on 52, you will get silent wrong predictions. Saving the window size in `model_config` and checking it in `validate_input()` catches this mistake immediately.

**Table 6.2 — What the pipeline saves for the CO₂ forecast model**

| What is saved | Why it matters |
|--------------|----------------|
| Model architecture config | Rebuilds the exact same network on load |
| `window_size` (in model_config) | `validate_input()` checks incoming sequence length |
| Model weights (`state_dict`) | The learned parameters |
| Fitted `MinMaxScaler` | New data must be scaled the same way as training data |
| Training history | Loss curves — useful for diagnosing and continuing training |

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# ModelPipeline — Chapter 6 version
#
# Identical to Chapters 3–5 with one extension:
#   validate_input() checks that incoming sequences have the correct shape
#   (batch, window_size, input_size), reading window_size from model_config.
# ─────────────────────────────────────────────────────────────────────────────

import dill

class ModelPipeline:
    """Bundles a trained PyTorch model with its preprocessor and metadata.
    Provides: save | load | validate_input | predict | retrain.
    Chapter 6 extension: validate_input checks sequence shape.
    """

    def __init__(self, model, model_config, preprocessor=None,
                 feature_names=None, feature_ranges=None, model_class=None):
        self.model            = model
        self.model_config     = model_config
        self.preprocessor     = preprocessor
        self.feature_names    = feature_names or []
        self.feature_ranges   = feature_ranges or {}
        self._model_class     = model_class
        self.model_class      = model_class.__name__ if model_class else type(model).__name__
        self.training_history = {'train_loss': [], 'val_loss': []}

    def save(self, path):
        cls_obj     = self._model_class if self._model_class is not None else type(self.model)
        class_bytes = dill.dumps(cls_obj)
        checkpoint  = {
            'model_class'       : self.model_class,
            'model_class_bytes' : class_bytes,
            'model_config'      : self.model_config,
            'state_dict'        : self.model.state_dict(),
            'preprocessor'      : self.preprocessor,
            'feature_names'     : self.feature_names,
            'feature_ranges'    : self.feature_ranges,
            'training_history'  : self.training_history,
            'pytorch_version'   : torch.__version__,
            'saved_at'          : datetime.datetime.now().isoformat(),
        }
        torch.save(checkpoint, path)
        print(f'Pipeline saved → {path}')
        print(f'  model_class : {self.model_class}')
        print(f'  history     : {len(self.training_history["val_loss"])} epochs recorded')
        print(f'  saved_at    : {checkpoint["saved_at"]}')

    @classmethod
    def load(cls, path, device=None):
        if device is None: device = torch.device('cpu')
        ckpt       = torch.load(path, map_location=device, weights_only=False)
        ModelClass = dill.loads(ckpt['model_class_bytes'])
        # Filter model_config to only the keys the constructor accepts
        # (model_config may also store window_size, which is not a constructor arg)
        arch_keys  = {'input_size', 'hidden_size', 'num_layers', 'dropout', 'output_size'}
        arch_cfg   = {k: v for k, v in ckpt['model_config'].items() if k in arch_keys}
        model      = ModelClass(**arch_cfg)
        model.load_state_dict(ckpt['state_dict'])
        model.to(device).eval()
        pipeline                  = cls(model=model, model_config=ckpt['model_config'],
                                        preprocessor=ckpt['preprocessor'],
                                        feature_names=ckpt['feature_names'],
                                        feature_ranges=ckpt['feature_ranges'])
        pipeline._model_class     = ModelClass
        pipeline.model_class      = ckpt['model_class']
        pipeline.training_history = ckpt['training_history']
        print(f'Pipeline loaded from {path}')
        print(f'  model_class : {pipeline.model_class}')
        print(f'  window_size : {pipeline.model_config.get("window_size", "not set")}')
        return pipeline

    def validate_input(self, X):
        """Check that X has shape (batch, window_size, input_size)."""
        expected_w = self.model_config.get('window_size')
        expected_f = self.model_config.get('input_size', 1)
        if not isinstance(X, torch.Tensor):
            raise ValueError('Input must be a torch.Tensor.')
        if X.ndim != 3:
            raise ValueError(f'Expected 3-D tensor (batch, window, features), got {X.shape}.')
        if expected_w is not None and X.shape[1] != expected_w:
            raise ValueError(f'Window size mismatch: model expects {expected_w}, got {X.shape[1]}.')
        if X.shape[2] != expected_f:
            raise ValueError(f'Feature mismatch: model expects {expected_f}, got {X.shape[2]}.')
        return True

    def predict(self, X_raw_np):
        """X_raw_np: numpy array (batch, window_size) — raw unscaled CO₂ values.
        Returns predictions in original CO₂ units (ppm).
        """
        if X_raw_np.ndim == 2:
            X_raw_np = X_raw_np[:, :, np.newaxis]
        # Apply preprocessor only if one was saved (e.g. StandardScaler)
        # If preprocessor is None (raw values), skip transform entirely
        if self.preprocessor is not None:
            flat = X_raw_np.reshape(-1, 1)
            X_in = self.preprocessor.transform(flat).reshape(X_raw_np.shape)
        else:
            X_in = X_raw_np
        X_t = torch.tensor(X_in.astype(np.float32))
        self.validate_input(X_t)
        self.model.eval()
        with torch.no_grad():
            preds = self.model(X_t).cpu().numpy()
        if self.preprocessor is not None:
            return self.preprocessor.inverse_transform(preds.reshape(-1, 1)).flatten()
        return preds.flatten()

    def retrain(self, train_loader, val_loader, criterion, optimiser, num_epochs, device):
        """Continue training; append results to training_history."""
        self.model.train()
        new_tr, new_va = run_training(
            self.model, train_loader, val_loader, criterion, optimiser,
            num_epochs=num_epochs, device=device, verbose_every=5,
        )
        self.training_history['train_loss'].extend(new_tr)
        self.training_history['val_loss'].extend(new_va)
        print(f'Retrain complete. Total epochs in history: {len(self.training_history["val_loss"])}')


print('ModelPipeline defined.')
print('Methods: save | load | validate_input | predict | retrain')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5a — Save pipeline
# ─────────────────────────────────────────────────────────────────────────────

co2_model_config = {
    **CO2_CONFIG,
    'window_size' : SEQ_LEN,
    'co2_mean'    : float(CO2_MEAN),
    'co2_std'     : float(CO2_STD),
    'years_used'  : YEARS_TO_USE,
}

pipeline = ModelPipeline(
    model         = co2_model,
    model_config  = co2_model_config,
    preprocessor  = None,
    feature_names = ['co2_zscore', 'week_idx_embed'],
    model_class   = LSTMWithWeekEmbed,
)
pipeline.training_history = {'train_loss': co2_train_hist, 'val_loss': co2_val_hist}
pipeline.save('co2_forecast_v1.pth')
print(f'Stored: window_size={SEQ_LEN}  co2_mean={CO2_MEAN:.2f}  co2_std={CO2_STD:.2f}')


In [ ]:
HORIZON = 52
loaded   = ModelPipeline.load('co2_forecast_v1.pth')
W        = loaded.model_config['window_size']
co2_mean = loaded.model_config['co2_mean']
co2_std  = loaded.model_config['co2_std']
print(f'Loaded  window_size={W}  co2_mean={co2_mean:.2f}  co2_std={co2_std:.2f}')

last_date    = daily.date.iloc[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(weeks=1), periods=HORIZON, freq='W')
future_week_idx = ((future_dates.dayofyear / 7).astype(int) % 52).astype(np.int64)

# Seed from daily['week_idx'] column — defined in the load cell
seed_co2_z  = ((daily.co2_ppm.values[-W:] - co2_mean) / co2_std).astype(np.float32)
seed_wk_idx = daily['week_idx'].values[-W:].astype(np.int64)

window_co2 = seed_co2_z.copy()
window_wk  = seed_wk_idx.copy()

forecast_z = []
loaded.model.eval()
with torch.no_grad():
    for step in range(HORIZON):
        co2_t  = torch.tensor(window_co2, dtype=torch.float32).unsqueeze(0).unsqueeze(-1)
        wk_t   = torch.tensor(window_wk,  dtype=torch.int64).unsqueeze(0)
        pred_z = loaded.model(co2_t.to(device), wk_t.to(device)).cpu().item()
        forecast_z.append(pred_z)
        window_co2 = np.append(window_co2[1:], pred_z)
        window_wk  = np.append(window_wk[1:],  future_week_idx[step])

forecast = np.array(forecast_z) * co2_std + co2_mean

fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')
context = daily.tail(3 * 52)
ax.plot(context.date, context.co2_ppm, color='#1F3864', lw=1.5, label='Actual (last 3 years)')
ax.plot(future_dates, forecast, color='#e74c3c', lw=2.0, linestyle='--', label='52-week forecast')
ax.axvline(x=last_date, color='#aaaaaa', lw=1.2, linestyle=':')
ax.set_ylabel('CO2 (ppm)', fontsize=10)
ax.set_title(f'Figure 6.7 — 52-week CO2 Forecast  (window_size={W} from .pth)',
             fontsize=11, fontweight='bold', color='#1F3864')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

actual_last = daily.co2_ppm.iloc[-1]
print(f'Week  1: {forecast[0]:.2f} ppm')
print(f'Week 26: {forecast[25]:.2f} ppm')
print(f'Week 52: {forecast[-1]:.2f} ppm')
print(f'Annual rise: {forecast[-1] - actual_last:+.2f} ppm  (historical ~+1.8 ppm)')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.5c — Retrain with new data and save v2
# Run cell 6.5b before this one.
# ─────────────────────────────────────────────────────────────────────────────

if 'loaded' not in dir():
    raise RuntimeError('Run cell 6.5b first.')

print(f'Retraining: window_size={loaded.model_config["window_size"]}  '
      f'co2_mean={loaded.model_config["co2_mean"]:.2f}  '
      f'co2_std={loaded.model_config["co2_std"]:.2f}  '
      f'(all from .pth)')

retrain_opt = optim.Adam(loaded.model.parameters(), lr=2e-4)

print('Retraining (10 epochs):')
for epoch in range(10):
    tl = train_embed(loaded.model, co2_train_loader, nn.MSELoss(), retrain_opt, device)
    vl = eval_embed( loaded.model, co2_val_loader,   nn.MSELoss(), device)
    loaded.training_history['train_loss'].append(tl)
    loaded.training_history['val_loss'].append(vl)
    if (epoch + 1) % 5 == 0:
        print(f'  Epoch {epoch+1}: train={tl:.4f}  val={vl:.4f}')

loaded.save('co2_forecast_v2.pth')
print(f'Total epochs: {len(loaded.training_history["val_loss"])}')


### 📝 Exercise 6.5 — Pipeline for Sequence Models

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Exercise 6.5
#
# Task A: Load co2_forecast_v1.pth and intentionally trigger a ValueError
#         by calling validate_input() with the wrong sequence length.
#         Confirm the error message is clear and informative.
#
# Task B: Load co2_forecast_v2.pth and generate a 52-week forecast.
#         Plot v1 and v2 forecasts on the same axes.
#         Do the two forecasts differ? Describe what changed.
#
# Task C: Answer as comments:
#   Q1. Why is it important to save the window_size inside model_config
#       rather than just as a separate variable in your script?
#   Q2. The forecast gets less reliable further into the future.
#       In one sentence, explain why (hint: think about what the model
#       is using as input after the first few predictions).
# ─────────────────────────────────────────────────────────────────────────────

# Task A
# v1 = ModelPipeline.load('co2_forecast_v1.pth')
# bad_input = torch.zeros(1, 30, 1)   # wrong: model expects window_size=52
# try:
#     v1.validate_input(bad_input)
# except ValueError as e:
#     print(f'Caught expected error: {e}')

# Task B
# v2 = ModelPipeline.load('co2_forecast_v2.pth')
# ... (generate forecast, plot both)

# Task C
# Q1. Answer: ?
# Q2. Answer: ?

# See the Answer Key at the end of this chapter.
print('Exercise 6.5 complete.')

---
# 6.6 ★ Bonus: Autoencoders

## What is an autoencoder?

Every model in this book so far has been **supervised**: it learns a mapping from inputs to labels. An autoencoder flips this around. It has no labels at all — its target is its own input.

The goal is to compress the input into a small **bottleneck** representation and then reconstruct the original from that compressed form. The network learns by minimising the difference between input and reconstruction.

```
Input x          Bottleneck z          Reconstruction x̂
  (20 features)    (4 features)           (20 features)

  ┌─────────┐       ┌─────┐              ┌─────────┐
  │  x₁     │       │     │              │  x̂₁    │
  │  x₂     │──────►│  z  │─────────────►│  x̂₂    │
  │  ...    │       │     │              │  ...    │
  │  x₂₀   │       └─────┘              │  x̂₂₀   │
  └─────────┘                           └─────────┘
    ENCODER          (compress)          DECODER

  Loss = MSE(x, x̂)   ← no labels needed
```

*Figure 6.8 — Autoencoder structure. The bottleneck forces the encoder to keep only the most essential information.*

## Why the bottleneck matters

Because the bottleneck has fewer dimensions than the input, the encoder is forced to discard anything that is not essential and keep only the most important structure. This is the same idea as compressing a photo into a smaller file — the compression removes redundancy while preserving the key content.

## Business applications

| Application | How the autoencoder is used |
|-------------|-----------------------------|
| **Anomaly detection** | Train on normal data. Anomalous inputs reconstruct poorly — high reconstruction error flags them |
| **Dimensionality reduction** | Use the bottleneck as a compressed feature set for downstream models |
| **Data denoising** | Train with noisy inputs and clean targets — the network learns to remove noise |
| **Feature learning** | Pre-train on unlabelled data; fine-tune on a small labelled set |

**Anomaly detection** is the most common business use. You train only on normal transactions, normal sensor readings, or normal images. The model learns to reconstruct "normal" well. When an anomaly arrives, it does not match any pattern the encoder has learned — reconstruction error is high. Flag it.

> **Why here, in Chapter 6?** You now have the full supervised toolkit: FFNs (Ch. 4), CNNs (Ch. 5), LSTMs (this chapter). Autoencoders use the same building blocks — `nn.Linear`, `nn.Conv2d`, `nn.LSTM` — but flip the task. This is the bridge to unsupervised learning and directly prepares you for Chapter 7, where large language models learn from unlabelled text using a related self-supervised idea.

### 🔬 Illustrations 6.6 — Autoencoders in Action

The two cells below are fully-runnable illustrations — just run them top to bottom.

| Part | Architecture | What you will see |
|------|-------------|-------------------|
| 1 | **CNN Autoencoder** | Portrait scratch removal: noisy input → clean restoration |
| 2 | **LSTM Autoencoder** | Time-series anomaly detection: normal sequences reconstruct cleanly; anomalous ones do not |


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6 Illustration 1 — CNN Autoencoder: photo scratch removal
#
# We use the built-in skimage astronaut portrait (public domain NASA photo).
# Three thick diagonal scratch lines are drawn across the face.
# A CNN Autoencoder learns to restore the clean image from the scratched one.
#
# What you will see:
#   Left  — Original clean portrait
#   Middle — Scratched (damaged) portrait
#   Right  — CNN restored portrait (scratches removed)
# ─────────────────────────────────────────────────────────────────────────────

!pip install -q scikit-image
from skimage import data as skdata, color, transform as sktransform

# ── Load and prepare portrait ─────────────────────────────────────────────────
IMG_SIZE = 128
portrait_rgb  = skdata.astronaut()                                       # (512,512,3) uint8
portrait_gray = color.rgb2gray(portrait_rgb).astype(np.float32)          # (512,512) [0,1]
portrait      = sktransform.resize(portrait_gray, (IMG_SIZE, IMG_SIZE),  # (128,128)
                                   anti_aliasing=True).astype(np.float32)

def add_scratches(img, n_lines=3, thickness=4):
    """Draw n thick diagonal white scratch lines across the image."""
    import cv2  # available on Colab
    scratch = (img * 255).astype(np.uint8)
    h, w = scratch.shape
    np.random.seed(7)   # fixed seed = same scratches every run
    for _ in range(n_lines):
        x0 = np.random.randint(0,  w // 3)
        y0 = np.random.randint(0,  h // 3)
        x1 = np.random.randint(2 * w // 3, w)
        y1 = np.random.randint(2 * h // 3, h)
        cv2.line(scratch, (x0, y0), (x1, y1), 255, thickness)
    return scratch.astype(np.float32) / 255.0

scratched = add_scratches(portrait, n_lines=3, thickness=4)

print(f'Portrait size  : {portrait.shape}  ({IMG_SIZE}×{IMG_SIZE} greyscale)')
print(f'Scratch pixels : {(scratched == 1.0).sum()}  '
      f'({(scratched == 1.0).mean()*100:.1f}% of image)')

# ── Create training dataset via augmentation ──────────────────────────────────
# We generate 1,000 slightly-varied versions of the same image:
#   random brightness jitter + random pixel noise
# This prevents the model from just memorising one example.
N_AUG = 1000
np.random.seed(SEED)

def augment(img, noise_std=0.02):
    gain   = 1.0 + np.random.uniform(-0.1, 0.1)   # ±10% brightness
    noisy  = np.clip(img * gain + noise_std * np.random.randn(*img.shape), 0, 1)
    return noisy.astype(np.float32)

clean_aug    = np.stack([augment(portrait)  for _ in range(N_AUG)])   # (N,128,128)
scratch_aug  = np.stack([augment(scratched) for _ in range(N_AUG)])   # (N,128,128)

# Shape: (N, 1, 128, 128)
clean_t   = torch.tensor(clean_aug[:,  np.newaxis], dtype=torch.float32)
scratch_t = torch.tensor(scratch_aug[:, np.newaxis], dtype=torch.float32)

sp = int(N_AUG * 0.85)
cnn_tr = DataLoader(TensorDataset(scratch_t[:sp], clean_t[:sp]),   batch_size=16, shuffle=True)
cnn_va = DataLoader(TensorDataset(scratch_t[sp:], clean_t[sp:]),   batch_size=16)
print(f'Augmented training pairs: {sp}   validation: {N_AUG - sp}')

# ── CNN Autoencoder for 128×128 images ───────────────────────────────────────
class CNNAutoencoder(nn.Module):
    """
    Encoder: (1,128,128) → (32,64,64) → (64,32,32) → (128,16,16)
    Decoder: (128,16,16) → (64,32,32) → (32,64,64) → (1,128,128)
    Three pooling levels give a large enough receptive field to see
    and fill in an entire scratch line in one pass.
    """
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 32,  3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 64×64
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 32×32
            nn.Conv2d(64,128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 16×16
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.ReLU(),           # 32×32
            nn.ConvTranspose2d(64,  32, 2, stride=2), nn.ReLU(),           # 64×64
            nn.ConvTranspose2d(32,   1, 2, stride=2), nn.Sigmoid(),        # 128×128
        )
    def forward(self, x): return self.decoder(self.encoder(x))


torch.manual_seed(SEED)
cnn_ae   = CNNAutoencoder().to(device)
cnn_opt  = optim.Adam(cnn_ae.parameters(), lr=1e-3)
cnn_crit = nn.MSELoss()
print(f'CNN AE parameters: {sum(p.numel() for p in cnn_ae.parameters()):,}')

# ── Train ─────────────────────────────────────────────────────────────────────
print('Training CNN Autoencoder — portrait scratch removal (40 epochs):')
best_val, best_state = float('inf'), None

for epoch in range(40):
    cnn_ae.train()
    for s_b, c_b in cnn_tr:
        cnn_opt.zero_grad()
        loss = cnn_crit(cnn_ae(s_b.to(device)), c_b.to(device))
        loss.backward(); cnn_opt.step()
    cnn_ae.eval()
    va = sum(cnn_crit(cnn_ae(s_b.to(device)), c_b.to(device)).item()
             for s_b, c_b in cnn_va) / len(cnn_va)
    if va < best_val:
        best_val   = va
        best_state = copy.deepcopy(cnn_ae.state_dict())
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:2d}: val loss = {va:.5f}')

cnn_ae.load_state_dict(best_state)   # restore best weights
print(f'Best val loss: {best_val:.5f}')

# ── Restore the ORIGINAL (non-augmented) scratched image ─────────────────────
cnn_ae.eval()
original_tensor  = torch.tensor(portrait[np.newaxis, np.newaxis],   dtype=torch.float32)
scratched_tensor = torch.tensor(scratched[np.newaxis, np.newaxis],  dtype=torch.float32)
with torch.no_grad():
    restored_tensor = cnn_ae(scratched_tensor.to(device)).cpu()

orig_img  = original_tensor[0, 0].numpy()
sctch_img = scratched_tensor[0, 0].numpy()
rest_img  = restored_tensor[0, 0].numpy()

# ── Figure 6.9: side-by-side comparison ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
fig.patch.set_facecolor('white')

titles = ['Original (clean)', 'Scratched (input to model)', 'Restored (model output)']
images = [orig_img, sctch_img, rest_img]
border = ['#27ae60', '#e74c3c', '#2980b9']

for ax, img, title, col in zip(axes, images, titles, border):
    ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    ax.set_title(title, fontsize=10, fontweight='bold', color=col, pad=8)
    ax.axis('off')
    for spine in ax.spines.values():
        spine.set_edgecolor(col); spine.set_linewidth(2)

fig.suptitle('Figure 6.9 — CNN Autoencoder: scratch removal on astronaut portrait\n'
             'The model was trained only on (scratched → clean) pairs — no labels needed',
             fontsize=10, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()

# Quality metrics
mse_before = np.mean((sctch_img - orig_img)**2)
mse_after  = np.mean((rest_img  - orig_img)**2)
print(f'MSE before restoration: {mse_before:.5f}')
print(f'MSE after  restoration: {mse_after:.5f}')
print(f'Improvement            : {(1 - mse_after/mse_before)*100:.1f}% reduction in error')


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 6.6 Illustration 2 — LSTM Autoencoder: time-series anomaly detection
#
# Scenario: daily power consumption follows a weekly sine pattern.
# On anomalous days a sudden spike or drop occurs.
# We train on NORMAL patterns only; anomalies flag themselves by
# reconstructing poorly (high MSE).
# ─────────────────────────────────────────────────────────────────────────────

class LSTMAutoencoder(nn.Module):
    """Encoder  : LSTM reads seq -> bottleneck h_n
       Decoder  : LSTM receives h_n (repeated) -> reconstructed sequence
       Loss     : MSELoss(x_hat, x)   target = input
    """
    def __init__(self, input_size=1, hidden_size=32, seq_len=30):
        super().__init__()
        self.seq_len      = seq_len
        self.encoder_lstm = nn.LSTM(input_size,  hidden_size, batch_first=True)
        self.decoder_lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.output_layer = nn.Linear(hidden_size, input_size)

    def forward(self, x):
        _, (h_n, c_n) = self.encoder_lstm(x)
        dec_in        = h_n[-1].unsqueeze(1).repeat(1, self.seq_len, 1)
        decoded, _    = self.decoder_lstm(dec_in, (h_n, c_n))
        return self.output_layer(decoded)


SEQ_AE = 30
np.random.seed(SEED)

def make_power(n, anomaly=False, strength=3.0):
    windows = []
    for _ in range(n):
        t   = np.arange(SEQ_AE)
        seq = np.sin(2 * np.pi * t / 7) + 0.1 * np.random.randn(SEQ_AE)
        if anomaly:
            pos = np.random.randint(5, SEQ_AE - 5)
            seq[pos] += strength * np.random.choice([-1, 1])
        windows.append(seq.astype(np.float32))
    return torch.tensor(np.array(windows)).unsqueeze(-1)  # (n, seq, 1)


X_norm = make_power(2000, anomaly=False)
X_anom = make_power(30,   anomaly=True)

sp = 1600
ld_tr = DataLoader(TensorDataset(X_norm[:sp], X_norm[:sp]), batch_size=64, shuffle=True)
ld_va = DataLoader(TensorDataset(X_norm[sp:], X_norm[sp:]), batch_size=64)

torch.manual_seed(SEED)
lstm_ae  = LSTMAutoencoder(seq_len=SEQ_AE).to(device)
lstm_opt = optim.Adam(lstm_ae.parameters(), lr=1e-3)
lstm_crit = nn.MSELoss()

print(f'LSTM AE params: {sum(p.numel() for p in lstm_ae.parameters()):,}')
print('Training LSTM Autoencoder on normal power sequences (25 epochs):')

for epoch in range(25):
    lstm_ae.train()
    for xb, _ in ld_tr:
        lstm_opt.zero_grad()
        loss = lstm_crit(lstm_ae(xb.to(device)), xb.to(device))  # target = input
        loss.backward()
        nn.utils.clip_grad_norm_(lstm_ae.parameters(), 1.0)
        lstm_opt.step()
    if (epoch + 1) % 5 == 0:
        lstm_ae.eval()
        va = sum(lstm_crit(lstm_ae(xb.to(device)), xb.to(device)).item()
                 for xb, _ in ld_va) / len(ld_va)
        print(f'  Epoch {epoch+1:2d}: val loss = {va:.5f}')

# ── Anomaly detection ─────────────────────────────────────────────────────────
lstm_ae.eval()
with torch.no_grad():
    err_n = ((lstm_ae(X_norm.to(device)) - X_norm.to(device))**2).mean(dim=[1,2]).cpu()
    err_a = ((lstm_ae(X_anom.to(device)) - X_anom.to(device))**2).mean(dim=[1,2]).cpu()

thresh  = err_n.mean() + 2 * err_n.std()
flagged = (err_a > thresh).sum().item()

print()
print(f'Recon error — normal    : mean={err_n.mean():.5f}  std={err_n.std():.5f}')
print(f'Recon error — anomalous : mean={err_a.mean():.5f}  std={err_a.std():.5f}')
print(f'Threshold (mean+2std)   : {thresh:.5f}')
print(f'Anomalies flagged       : {flagged} / {len(X_anom)}')

# ── Visualise a normal vs anomalous window and their reconstructions ───────────
with torch.no_grad():
    r_norm = lstm_ae(X_norm[:1].to(device)).cpu().squeeze().numpy()
    r_anom = lstm_ae(X_anom[:1].to(device)).cpu().squeeze().numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('white')
t = np.arange(SEQ_AE)

for ax, orig, recon, title, col in zip(
    axes,
    [X_norm[0].squeeze().numpy(), X_anom[0].squeeze().numpy()],
    [r_norm,  r_anom],
    ['Normal sequence (low reconstruction error)',
     'Anomalous sequence (high reconstruction error — spike at day ~15)'],
    ['#27ae60', '#e74c3c'],
):
    ax.set_facecolor('white')
    ax.plot(t, orig,  color='#1F3864', lw=1.5, label='Original')
    ax.plot(t, recon, color=col,       lw=1.5, label='Reconstructed', linestyle='--')
    ax.set_title(title, fontsize=9, fontweight='bold', color='#1F3864')
    ax.set_xlabel('Day'); ax.set_ylabel('Power'); ax.legend(fontsize=8)

fig.suptitle('Figure 6.10 — LSTM Autoencoder: normal data reconstructs cleanly; '
             'anomalous spike leaves a large gap',
             fontsize=11, fontweight='bold', color='#1F3864')
plt.tight_layout()
plt.show()


---
## Chapter 6 Summary

| Concept | Key takeaway |
|---------|-------------|
| **Why sequences are different** | Order carries meaning — the past shapes what comes next; FFNs treat every input as independent |
| **Hidden state** | The network's running memory — updated at every step, carries context forward |
| **Parameter sharing** | The same RNN cell (same weights) is applied at every time step — no extra parameters for longer sequences |
| **Vanilla RNN** | `nn.RNN` — simple, fast, but gradients vanish through long sequences; fails beyond ~20 steps |
| **Vanishing gradient** | Like a whispered message through 50 people — the original signal is garbled by the time it reaches the end |
| **LSTM** | `nn.LSTM` — two memory channels (hidden state + cell state); additive cell updates preserve gradients |
| **Four LSTM computations** | Three sigmoid gates (forget $f_t$, input $i_t$, output $o_t$) + one tanh candidate ($\tilde{c}_t$). Often called "three gates" when the candidate is bundled with input — but all four have their own weight matrices, explaining the ~4× parameter cost |
| **Gradient clipping** | `clip_grad_norm_(max_norm=1.0)` — prevents rare but catastrophic gradient explosions in RNNs/LSTMs |
| **Sliding window** | Converts a flat series into (X, y) pairs; always use a temporal split — random splits leak the future |
| **MAE and MAPE** | MAE in original units; MAPE as a percentage — MAPE is preferred when comparing across different scales |
| **Mauna Loa CO₂** | Built into `statsmodels`; strong trend + yearly seasonality; LSTM learns both patterns visibly |
| **Autoencoder** | Unsupervised — target is the input itself; encoder compresses, decoder reconstructs; high error = anomaly |
| **`LSTMWithWeekEmbed`** | Extends `LSTMNet` with a week embedding; forward pass takes `(co2_z, week_idx)` — two separate inputs |
| **ModelPipeline (sequence)** | Same contract as Ch. 3–5; stores `window_size` in `model_config`; `validate_input()` checks tensor shape |

---

## What Comes Next

Chapter 7 introduces **Large Language Models** — the architecture that has transformed every industry that works with text. Where the LSTM in this chapter learned patterns from 2,000 weekly numbers, an LLM arrives pre-trained on hundreds of billions of tokens. Rather than train a model, Chapter 7 shows how to *direct* one: through prompt engineering, structured outputs, and a full capstone project building a business Q&A bot over a corpus of earnings call transcripts.

---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*

---
## Answer Key — Chapter 6 Exercises

Use these suggested answers to check your thinking. For open-ended exercises (6.1) there is no single correct answer — the sample responses show the kind of reasoning to aim for.

---

### Exercise 6.1 — Sequence Problems in Your Domain

*Open-ended. Sample responses:*

1. **Daily hospital admissions**  
   Sequence: daily emergency admission counts | Target: next week's volume | Why: flu seasons, public holidays, and heatwaves create temporal patterns — today's count depends heavily on the last 14 days

2. **Customer support ticket volume**  
   Sequence: hourly ticket count | Target: next hour's volume | Why: Monday morning spikes, lunch dips, and product-launch surges are all temporal; an FFN ignoring time would miss them entirely

3. **Retail energy consumption**  
   Sequence: hourly kilowatt readings per store | Target: next day's consumption | Why: opening hours, day of week, and seasonal heating/cooling create strong time-dependent patterns

---

### Exercise 6.3 — Replace RNN with LSTM

**Q1.** On short sequences (seq_len=10), the RNN and LSTM perform comparably — there are only 10 gradient multiplications, so vanishing is not an issue. For short sequences, prefer the RNN: fewer parameters, faster to train, equally accurate. Use LSTM when seq_len > ~20.

**Q2.** No. For short sequences the extra parameters add training time without adding accuracy. The ~4× parameter cost of an LSTM is only justified when the task requires long-range memory.

```python
torch.manual_seed(SEED)
X_short, y_short = make_sine_dataset(SHORT)
tr_short, va_short = make_loaders(X_short, y_short)
lstm_short = LSTMNet(input_size=1, hidden_size=32, num_layers=1, dropout=0.0).to(device)
_, lstm_short_val = run_training(
    lstm_short, tr_short, va_short, nn.MSELoss(),
    optim.Adam(lstm_short.parameters(), lr=1e-3),
    num_epochs=40, device=device, verbose_every=None)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(rnn_results[f'RNN  seq={SHORT}'], label=f'RNN  seq={SHORT}', color='#27ae60', lw=2)
ax.plot(lstm_short_val,                   label=f'LSTM seq={SHORT}', color='#2980b9', lw=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Val MSE'); ax.legend()
plt.tight_layout(); plt.show()
# Both converge similarly — LSTM advantage only appears on long sequences.
```

---

### Exercise 6.4 — End-to-End CO₂ Forecasting

**Task A — shorter window (SEQ_LEN=26):**  
MAE typically worsens with a 26-week window. The yearly seasonal cycle takes 52 weeks to complete. A 26-week window only sees half a cycle — the model cannot observe a full summer dip followed by a winter rise in a single input window, so it learns the pattern less precisely.

```python
SEQ_LEN_A = 26
X_a, y_a = make_windows(scaled, SEQ_LEN_A)
tr_split_a = split - SEQ_LEN_A
X_tr_a, y_tr_a = X_a[:tr_split_a], y_a[:tr_split_a]
X_va_a, y_va_a = X_a[tr_split_a:], y_a[tr_split_a:]
# ... (same training loop; compare final MAE)
```

**Task B — smaller model (hidden_size=64):**  
MAE increases slightly. The embedding + 2-layer LSTM already has good capacity at hidden=128; reducing to 64 still learns the seasonal shape but may miss finer-grained trend patterns.

**Q1 — Why is a random split wrong?**  
A random split places some future observations in the training set. The model sees the future during training — this is data leakage. At deployment no future data exists. A temporal split honestly simulates the real-world scenario: train on everything before time $t$, evaluate on everything after.

**Q2 — Why is MAPE useful here?**  
MAPE expresses error as a percentage of the actual value, making it scale-independent. This is useful when comparing forecast quality across different series (e.g., a series ranging 300–370 vs one ranging 0–100). For CO₂, MAPE tells you what fraction of the typical reading you are missing — intuitive for both scientists and policymakers.

---

### Exercise 6.5 — Pipeline for Sequence Models

**Task A — trigger a validation error:**
```python
v1 = ModelPipeline.load('co2_forecast_v1.pth')
bad_input = torch.zeros(1, 30, 1)   # wrong: model expects window_size=52
try:
    v1.validate_input(bad_input)
except ValueError as e:
    print(f'Caught expected error: {e}')
# Output: Window size mismatch: model expects 52, got 30.
```

**Q1 — Why store window_size inside model_config?**  
A separate variable in your script can easily be changed or forgotten. Storing it inside the `.pth` file means anyone loading the model on any machine automatically has the correct window size — the pipeline is self-contained.

**Q2 — Why does forecast quality degrade over time?**  
After the first few steps, the model is feeding its own (imperfect) predictions back as inputs. Errors compound: a small mistake at step 5 becomes the input at step 6, making step 6 slightly worse, and so on across 52 steps. The further into the future, the more the model is forecasting from its own noise rather than real data.



---
*Deep Learning for Business Analytics: From Basics to Large Language Models*  
*Dr. M. Ramasubramaniam &amp; Mr. Daniel Peter*